# RSA — 1-bit / BBQ-inspired semantic predicate experiment

This is the quality kill-test for the low-bit semantic-substrate idea. It compares:

- FP32 supervised linear semantic heads
- PQ64 + compiled linear LUTs
- **BBQ-inspired centered 1-bit documents + two per-item correction values**, with FP32 and int4 predicate weights
- **RSA1 centered identity** sparse learned predicates (48 B/item)
- **RSA1 centered random** sparse learned predicates (48 B/item)
- RSA2 random (96 B/item)
- RSA4 random (192 B/item)
- zero-shot MiniLM, dense ordering, and oracle

The BBQ-inspired baseline is intentionally labeled as such: it isolates globally centered 1-bit documents, asymmetric int4 predicate weights, and small corrective state, but is not a byte-for-byte Lucene BBQ implementation.


In [ ]:
#@title 1) Settings
FULL_RUN = False #@param {type:"boolean"}
RUN_TESTS = True #@param {type:"boolean"}
print('FULL_RUN =', FULL_RUN)


In [ ]:
#@title 2) Clone public repo and install
import os, pathlib, shutil, subprocess
ROOT=pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run(['pip','install','-q','-e','.','faiss-cpu'],check=True)
print('repo:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())


In [ ]:
#@title 3) Tests
import subprocess, os
os.chdir('/content/ras')
if RUN_TESTS:
    subprocess.run(['pytest','-q','tests/test_binary.py','tests/test_smoke.py'],check=True)
else:
    print('tests skipped')


In [ ]:
#@title 4) Run binary / BBQ-inspired quality experiment
import os, subprocess, time
os.chdir('/content/ras')
cfg='configs/binary_bbq.yaml' if FULL_RUN else 'configs/binary_bbq_smoke.yaml'
print('config:',cfg)
t0=time.time()
subprocess.run(['python','-m','experiments.binary_bbq_predicates','--config',cfg],check=True)
print(f'finished in {(time.time()-t0)/60:.1f} min')


In [ ]:
#@title 5) Show headline + predicate quality
from pathlib import Path
import json, pandas as pd
root=Path('/content/ras/results')
runs=sorted([p for p in root.iterdir() if p.is_dir() and '_binary_' in p.name],key=lambda p:p.stat().st_mtime)
run=runs[-1]
print('run:',run)
headline=json.loads((run/'headline.json').read_text())
print(json.dumps(headline,indent=2))
pred=pd.read_csv(run/'predicate_summary.csv').sort_values('f1',ascending=False)
display(pred)


In [ ]:
#@title 6) Quality × item bytes × predicate bytes at 20% retention
import pandas as pd
pareto=pd.read_csv(run/'pareto_at_20pct.csv').sort_values('recall',ascending=False)
display(pareto[['method','bytes_per_item','program_bytes_per_concept','recall','purity','notes']])

# Compact decision table
focus=['linear_fp32','pq64_linear_lut','bbq1_ls2_f32q','bbq1_ls2_int4q','rsa1_centered_identity','rsa1_centered_random','rsa2_random','rsa4_random']
display(pareto[pareto.method.isin(focus)][['method','bytes_per_item','program_bytes_per_concept','recall','purity']].sort_values('bytes_per_item'))


In [ ]:
#@title 7) Show full candidate-budget curves
from IPython.display import display, Image
for p in sorted((run/'figures').glob('*.png')):
    print(p.name)
    display(Image(filename=str(p)))


## What decides the next step?

The key comparisons are **RSA1 vs RSA4 vs PQ64 vs BBQ-inspired int4**.

- If RSA1 retains most of RSA4 quality, the next experiment is the Rust popcount/mask executor.
- If BBQ-inspired int4 dominates the learned RSA1 program, we should treat it as the stronger binary baseline and ask whether RSA adds value through nonlinearity/composition/program sparsity.
- If 1-bit quality collapses, 2-bit becomes the likely systems sweet spot.

The run also writes `native_export_first_seed.npz` containing packed real test codes and int4 predicate bitplanes for the native Rust follow-up.


In [ ]:
#@title 8) Zip all outputs
import shutil
zip_path=shutil.make_archive('/content/rsa_binary_bbq_experiment','zip',root_dir=str(run))
print(zip_path)
